# Structural Networks — Interactive 3D Visualization (Synapse Count)

Loads pre-built NetworkX graphs (`data/G_906_count.pkl`, `data/G_93_count.pkl`) where edge weights are **number of synapses per pair** rather than synaptic cleft size. Renders interactive Plotly figures you can rotate, zoom, and hover.

Also exports node and edge tables as CSV.

In [1]:
import pickle
import numpy as np
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import os

---
## Load Graphs

In [2]:
with open('data/G_906_count.pkl', 'rb') as f:
    G_906 = pickle.load(f)
with open('data/G_93_count.pkl', 'rb') as f:
    G_93 = pickle.load(f)

print(f'G_906: {G_906.number_of_nodes()} nodes, {G_906.number_of_edges():,} edges')
print(f'G_93:  {G_93.number_of_nodes()} nodes,  {G_93.number_of_edges():,} edges')

G_906: 906 nodes, 11,822 edges
G_93:  93 nodes,  229 edges


---
## Plot Helper

In [3]:
AREA_COL  = {'V1': '#4E79A7', 'RL': '#F28E2B', 'AL': '#59A14F',
             'LM': '#E15759', 'unknown': '#aaaaaa'}
LAYER_COL = {'L2/3': '#76B7B2', 'L4': '#EDC948', 'L5': '#B07AA1',
             'L6': '#FF9DA7', 'unknown': '#aaaaaa'}

def plot_3d(G, title, color_attr='brain_area', size_by='strength',
            edge_pct_keep=50, dark=True):
    # color_attr: 'brain_area' | 'layer' | 'pref_ori'
    # size_by:    'strength' (total # synapses) | 'degree' (# partners)
    # edge_pct_keep: keep top X% edges by weight (synapse count)
    nodes = list(G.nodes())
    xs = np.array([G.nodes[n]['x'] for n in nodes], dtype=float)
    ys = np.array([G.nodes[n]['y'] for n in nodes], dtype=float)
    zs = np.array([G.nodes[n]['z'] for n in nodes], dtype=float)

    if size_by == 'strength':
        raw = np.array([G.degree(n, weight='weight') for n in nodes], dtype=float)
    else:
        raw = np.array([G.degree(n) for n in nodes], dtype=float)
    norm_sz = (raw - raw.min()) / (raw.max() - raw.min() + 1e-9)
    sizes   = 4 + 16 * np.sqrt(norm_sz)

    raw_attr = [G.nodes[n].get(color_attr) for n in nodes]
    if color_attr == 'pref_ori':
        cvals = np.array([float(v) if v is not None else 0.0 for v in raw_attr])
        marker = dict(
            size=sizes, color=cvals, colorscale='HSV', cmin=0, cmax=180,
            colorbar=dict(title='Pref. ori. (deg)', thickness=14,
                          tickvals=[0, 45, 90, 135, 180],
                          tickfont=dict(color='white')),
            line=dict(width=0.8, color='rgba(255,255,255,0.5)'),
            opacity=0.92,
        )
        palette = {}
    else:
        palette = dict(AREA_COL if color_attr == 'brain_area' else LAYER_COL)
        unique_vals = sorted(set(str(v) for v in raw_attr))
        auto = px.colors.qualitative.Plotly
        for i, v in enumerate(unique_vals):
            if v not in palette:
                palette[v] = auto[i % len(auto)]
        col_list = [palette.get(str(v), '#aaaaaa') for v in raw_attr]
        marker = dict(
            size=sizes, color=col_list,
            line=dict(width=0.8, color='rgba(255,255,255,0.5)'),
            opacity=0.92,
        )

    hover = []
    for n in nodes:
        d   = G.nodes[n]
        st  = G.degree(n, weight='weight')  # total # synapses
        dg  = G.degree(n)                    # # partners
        ori = d.get('pref_ori')
        ori_s = f'{float(ori):.1f}' if ori is not None else 'N/A'
        hover.append(
            f'<b>Neuron {n}</b><br>'
            f'Area: {d.get("brain_area","?")}  Layer: {d.get("layer","?")}'
            f'  Type: {d.get("cell_type","?")}<br>'
            f'Pref ori: {ori_s} deg  gOSI: {float(d.get("gOSI",0)):.3f}<br>'
            f'<b>Partners: {dg}   # Synapses: {st}</b>'
        )

    node_trace = go.Scatter3d(
        x=xs, y=ys, z=zs, mode='markers',
        marker=marker, text=hover, hoverinfo='text',
        name='Neurons', showlegend=False,
    )

    legend_traces = []
    if palette:
        for val in sorted(palette):
            legend_traces.append(go.Scatter3d(
                x=[None], y=[None], z=[None], mode='markers',
                marker=dict(size=10, color=palette[val]),
                name=str(val), showlegend=True,
            ))

    all_edges = list(G.edges(data=True))
    edge_traces = []
    if all_edges:
        ew = np.array([d['weight'] for _, _, d in all_edges], dtype=float)
        threshold = np.percentile(ew, 100 - edge_pct_keep)
        sel = [(u, v, d) for u, v, d in all_edges if d['weight'] >= threshold]
        ew_sel  = np.array([d['weight'] for _, _, d in sel], dtype=float)
        ew_norm = (ew_sel - ew_sel.min()) / (ew_sel.max() - ew_sel.min() + 1e-9)
        N = 5
        buckets = [[] for _ in range(N)]
        for idx, (u, v, _) in enumerate(sel):
            b = min(int(ew_norm[idx] * N), N - 1)
            buckets[b].append((u, v))
        for b, elist in enumerate(buckets):
            if not elist:
                continue
            alpha = 0.05 + 0.25 * (b / (N - 1))
            ex, ey, ez = [], [], []
            for u, v in elist:
                ex += [G.nodes[u]['x'], G.nodes[v]['x'], None]
                ey += [G.nodes[u]['y'], G.nodes[v]['y'], None]
                ez += [G.nodes[u]['z'], G.nodes[v]['z'], None]
            edge_traces.append(go.Scatter3d(
                x=ex, y=ey, z=ez, mode='lines',
                line=dict(color=f'rgba(180,180,220,{alpha:.2f})', width=1),
                hoverinfo='none',
                showlegend=(b == N - 1),
                name=f'Pairs ({len(sel):,} shown)',
            ))

    bg   = '#0e0e12' if dark else 'white'
    axcol= '#888888' if dark else '#333'
    tcol = 'white'   if dark else 'black'

    fig = go.Figure(data=edge_traces + legend_traces + [node_trace])
    fig.update_layout(
        title=dict(text=title, x=0.5,
                   font=dict(size=19, color=tcol, family='Arial')),
        paper_bgcolor=bg,
        scene=dict(
            bgcolor=bg,
            xaxis=dict(title='x (nm)', showgrid=False, zeroline=False,
                       color=axcol, showbackground=False),
            yaxis=dict(title='y (nm)', showgrid=False, zeroline=False,
                       color=axcol, showbackground=False),
            zaxis=dict(title='z (nm)', showgrid=False, zeroline=False,
                       color=axcol, showbackground=False),
            camera=dict(eye=dict(x=1.4, y=1.4, z=0.8)),
        ),
        legend=dict(
            font=dict(color=tcol, size=12),
            bgcolor='rgba(0,0,0,0.35)',
            bordercolor='#444', borderwidth=1,
            itemsizing='constant',
        ),
        margin=dict(l=0, r=0, b=0, t=60),
        height=720,
    )
    return fig

---
## Full 906-Neuron Network

Colour = **brain area** (V1 / RL / AL / LM).
Node size ∝ total **number of synapses** (in + out).
Top 50% of pairs by synapse count drawn; brighter = more synapses.

In [4]:
fig_906 = plot_3d(
    G_906,
    title='Full 906-Neuron Network (synapse counts)  |  Colour: Brain Area',
    color_attr='brain_area',
    size_by='strength',
    edge_pct_keep=50,
)
fig_906.show()

Same network coloured by **cortical layer**, sized by **degree** (number of partners).

In [5]:
fig_906_layer = plot_3d(
    G_906,
    title='Full 906-Neuron Network (synapse counts)  |  Colour: Cortical Layer',
    color_attr='layer',
    size_by='degree',
    edge_pct_keep=30,
)
fig_906_layer.show()

---
## Session 9.3 V1 Subnetwork (93 Neurons)

All neurons are L4 in V1, so area/layer give no contrast.
Colour = **preferred orientation** (cyclic HSV, 0°–180°).
All pairs shown; edge brightness ∝ # synapses per pair.

In [6]:
fig_93 = plot_3d(
    G_93,
    title='Session 9.3 V1 Network (synapse counts)  |  Colour: Preferred Orientation',
    color_attr='pref_ori',
    size_by='degree',
    edge_pct_keep=100,
)
fig_93.show()

Same 93-neuron network, sized by **strength** (total # synapses), coloured by area.

In [7]:
fig_93b = plot_3d(
    G_93,
    title='Session 9.3 V1 Network (synapse counts)  |  Colour: Brain Area  |  Size: Strength',
    color_attr='brain_area',
    size_by='strength',
    edge_pct_keep=100,
)
fig_93b.show()

---
## Export Node and Edge Tables as CSV

Saves to `data/exports/`. One node CSV and one edge CSV per network. Edge weights are `n_synapses` (count per pair).

In [8]:
os.makedirs('data/exports', exist_ok=True)

def export_graph(G, prefix):
    rows = []
    for n in G.nodes():
        d = G.nodes[n]
        rows.append({
            'neuron_id':    n,
            'brain_area':   d.get('brain_area'),
            'layer':        d.get('layer'),
            'cell_type':    d.get('cell_type'),
            'pref_ori':     d.get('pref_ori'),
            'gOSI':         d.get('gOSI'),
            'x':            d.get('x'),
            'y':            d.get('y'),
            'z':            d.get('z'),
            'out_degree':   G.out_degree(n),
            'in_degree':    G.in_degree(n),
            'out_strength': G.out_degree(n, weight='weight'),
            'in_strength':  G.in_degree(n, weight='weight'),
        })
    nodes_df = pd.DataFrame(rows)
    nodes_df.to_csv(f'data/exports/{prefix}_nodes.csv', index=False)
    print(f'Saved data/exports/{prefix}_nodes.csv  ({len(nodes_df)} rows)')

    edges = [(u, v, G[u][v]['weight']) for u, v in G.edges()]
    edges_df = pd.DataFrame(edges,
                             columns=['pre_neuron_id', 'post_neuron_id', 'n_synapses'])
    edges_df.to_csv(f'data/exports/{prefix}_edges.csv', index=False)
    print(f'Saved data/exports/{prefix}_edges.csv  ({len(edges_df)} rows)')

export_graph(G_906, 'G_906_count')
export_graph(G_93,  'G_93_count')

Saved data/exports/G_906_count_nodes.csv  (906 rows)
Saved data/exports/G_906_count_edges.csv  (11822 rows)
Saved data/exports/G_93_count_nodes.csv  (93 rows)
Saved data/exports/G_93_count_edges.csv  (229 rows)


---
## Quick Stats Recap

In [9]:
for label, G in [('906-neuron', G_906), ('93-neuron (sess 9.3 V1)', G_93)]:
    od  = np.array([d for _, d in G.out_degree()])
    id_ = np.array([d for _, d in G.in_degree()])
    os_ = np.array([d for _, d in G.out_degree(weight='weight')])
    is_ = np.array([d for _, d in G.in_degree(weight='weight')])
    print(f"{'='*52}")
    print(f' {label}')
    print(f"{'='*52}")
    print(f'  Nodes:          {G.number_of_nodes()}')
    print(f'  Edges (pairs):  {G.number_of_edges():,}')
    print(f'  Density:        {nx.density(G):.4f}')
    print(f'  Mean out-deg:   {od.mean():.2f}   Max: {od.max()}')
    print(f'  Mean in-deg:    {id_.mean():.2f}   Max: {id_.max()}')
    print(f'  Mean out-strength (# syn): {os_.mean():.2f}   Max: {os_.max()}')
    print(f'  Mean in-strength (# syn):  {is_.mean():.2f}   Max: {is_.max()}')
    print()

 906-neuron
  Nodes:          906
  Edges (pairs):  11,822
  Density:        0.0144
  Mean out-deg:   13.05   Max: 69
  Mean in-deg:    13.05   Max: 98
  Mean out-strength (# syn): 13.05   Max: 69
  Mean in-strength (# syn):  13.05   Max: 98

 93-neuron (sess 9.3 V1)
  Nodes:          93
  Edges (pairs):  229
  Density:        0.0268
  Mean out-deg:   2.46   Max: 8
  Mean in-deg:    2.46   Max: 11
  Mean out-strength (# syn): 2.46   Max: 8
  Mean in-strength (# syn):  2.46   Max: 11

